# Install + imports + global style

In [1]:
# Install (Colab)
import sys
!{sys.executable} -m pip install -U nibabel torchio pandas matplotlib tqdm

# Imports
import os, json, time, random, glob, shutil, math
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchio as tio
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm import tqdm

# Professional, consistent look
mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
    "axes.titlepad": 8,
})


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.0/194.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 145.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 165.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 11.0 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.0.0
    Uninstalling wrapt-2.0.0:
      Successfully uninstalled wrapt-2.0.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully u

# Mount Drive & paths

In [2]:
# Mount Google Drive (Colab)
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not in Colab environment.")

# === Your project layout ===
PROJECT_DIR = "/content/drive/MyDrive/Brain_Tumor_Segmentation"
DATA_DIR = os.path.join(
    PROJECT_DIR,
    "ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
)
EXPERIMENT_NAME = "DHW_baseline"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, f"{EXPERIMENT_NAME}_checkpoints")

# Logs/metrics
METRICS_CSV              = os.path.join(CHECKPOINT_DIR, "epoch_metrics.csv")
TB_LOG_DIR               = os.path.join(CHECKPOINT_DIR, "tb_logs")
TEST_METRICS_CSV         = os.path.join(CHECKPOINT_DIR, "test_metrics.csv")
TEST_METRICS_PER_SUBJECT = os.path.join(CHECKPOINT_DIR, "test_metrics_per_subject.csv")

# Output
FIG_DIR = os.path.join(CHECKPOINT_DIR, "paper_figures")
os.makedirs(FIG_DIR, exist_ok=True)

# Optional predictions dir (if you saved per-subject predictions)
PRED_DIR = os.path.join(CHECKPOINT_DIR, "predictions")  # change if different

# Repro
SEED = 1337
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PROJECT_DIR     :", PROJECT_DIR)
print("DATA_DIR        :", DATA_DIR)
print("CHECKPOINT_DIR  :", CHECKPOINT_DIR)
print("FIG_DIR         :", FIG_DIR)
print("Device          :", device)


Mounted at /content/drive
PROJECT_DIR     : /content/drive/MyDrive/Brain_Tumor_Segmentation
DATA_DIR        : /content/drive/MyDrive/Brain_Tumor_Segmentation/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData
CHECKPOINT_DIR  : /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints
FIG_DIR         : /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/paper_figures
Device          : cuda


# Color maps & small viz helpers

In [3]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from skimage.measure import find_contours
import numpy as np

# Global style: bold titles, black canvas
mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "font.family": "DejaVu Sans",     # reproducible on Colab
    "axes.titleweight": "bold",
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Reference-like colors (WT=cyan, TC=yellow, ET=magenta)
cmap_wt = ListedColormap([[0,0,0,0], [0.00, 1.00, 0.00, 1.00]])  # green
cmap_tc = ListedColormap([[0,0,0,0], [1.00, 1.00, 0.00, 1.00]])  # yellow
cmap_et = ListedColormap([[0,0,0,0], [1.00, 0.00, 0.00, 1.00]])  # red

def _global_window(vols):
    """Compute a single (lo,hi) window shared by all modality panels for *this subject*.
       Here we use FLAIR percentiles; you can switch to a stacked array of all modalities."""
    ref = vols["t2f"].ravel()
    lo, hi = np.percentile(ref, [1, 99])
    return float(lo), float(hi)

def show_bg(ax, img, lo, hi):
    ax.set_facecolor("black")
    ax.imshow(img, cmap="gray", vmin=lo, vmax=hi, interpolation="none")
    ax.set_axis_off()

def _draw_contour(ax, mask2d, color, lw=1.5):
    # thin outline at value 0.5 to outline ROI
    for c in find_contours(mask2d.astype(float), 0.5):
        ax.plot(c[:,1], c[:,0], color=color, linewidth=lw, solid_capstyle="round")

def overlay_masks(ax, wt, tc, et, with_contours=False):
    # Solid fills
    ax.imshow(wt.astype(np.uint8), cmap=cmap_wt, vmin=0, vmax=1, interpolation="none")
    ax.imshow(tc.astype(np.uint8), cmap=cmap_tc, vmin=0, vmax=1, interpolation="none")
    ax.imshow(et.astype(np.uint8), cmap=cmap_et, vmin=0, vmax=1, interpolation="none")
    # Thin contours to match the “pop” in your reference
    """
    if with_contours:
        _draw_contour(ax, wt, "#00BFFF", lw=1.2)  # cyan-ish
        _draw_contour(ax, tc, "#E0B000", lw=1.2)  # yellow-ish
        _draw_contour(ax, et, "#E020D8", lw=1.2)  # magenta-ish
    """
"""
def mini_legend(ax, loc=(0.70, 0.72), sw=(0.065, 0.055)):
    # Small swatch legend in axes fraction coords.
    labels = [("WT", (0.00,0.75,1.00)),
              ("TC", (1.00,0.85,0.00)),
              ("ET", (0.95,0.20,0.90))]

    x0,y0 = loc
    w,h = sw
    for i,(txt, rgb) in enumerate(labels):
        ax.add_patch(plt.Rectangle((x0, y0-0.085*i), w, h,
                                   transform=ax.transAxes, facecolor=rgb,
                                   edgecolor="black", linewidth=0.4))
        ax.text(x0+w+0.02, y0-0.085*i+0.01, txt, transform=ax.transAxes,
                color="white", fontsize=9, weight="bold",
                va="bottom", ha="left")

"""

'\ndef mini_legend(ax, loc=(0.70, 0.72), sw=(0.065, 0.055)):\n    # Small swatch legend in axes fraction coords.\n    labels = [("WT", (0.00,0.75,1.00)),\n              ("TC", (1.00,0.85,0.00)),\n              ("ET", (0.95,0.20,0.90))]\n\n    x0,y0 = loc\n    w,h = sw\n    for i,(txt, rgb) in enumerate(labels):\n        ax.add_patch(plt.Rectangle((x0, y0-0.085*i), w, h,\n                                   transform=ax.transAxes, facecolor=rgb,\n                                   edgecolor="black", linewidth=0.4))\n        ax.text(x0+w+0.02, y0-0.085*i+0.01, txt, transform=ax.transAxes,\n                color="white", fontsize=9, weight="bold",\n                va="bottom", ha="left")\n\n'

# Data loaders & helpers

In [4]:
def subject_paths(root_dir, subject_id):
    base = os.path.join(root_dir, subject_id, subject_id)
    t1c = base + "-t1c.nii.gz"
    t1n = base + "-t1n.nii.gz"
    t2w = base + "-t2w.nii.gz"
    t2f = base + "-t2f.nii.gz"
    seg = base + "-seg.nii.gz"
    return dict(t1c=t1c, t1n=t1n, t2w=t2w, t2f=t2f, seg=seg)

def zscore_nonzero(x: np.ndarray) -> np.ndarray:
    mask = x > 0
    if mask.sum() > 0:
        m = x[mask].mean()
        s = x[mask].std()
        if s > 0:
            x = x.copy()
            x[mask] = (x[mask] - m) / s
        else:
            x = (x - x.mean()) / (x.std() + 1e-8)
    else:
        x = (x - x.mean()) / (x.std() + 1e-8)
    return x.astype(np.float32)

def load_modalities(subject_id):
    p = subject_paths(DATA_DIR, subject_id)
    vols = {}
    for k in ["t1c","t1n","t2w","t2f"]:
        vol = nib.load(p[k]).get_fdata().astype(np.float32)
        vols[k] = zscore_nonzero(vol)  # (H,W,D)
    seg = nib.load(p["seg"]).get_fdata().astype(np.int16)  # (H,W,D)
    return vols, seg

def brats_channels_from_seg(seg_hwd: np.ndarray):
    # 0=bg, 1=NCR/NET, 2=ED, 4=ET (some sets also use 3)
    et = ((seg_hwd == 3) | (seg_hwd == 4)).astype(np.uint8)
    tc = ((seg_hwd == 1) | (seg_hwd == 3)).astype(np.uint8)
    wt = (seg_hwd > 0).astype(np.uint8)
    return np.stack([wt, tc, et], axis=0)  # (3,H,W,D)  [WT,TC,ET]

def slice_with_max_tumor(mask_3ch_hwd: np.ndarray):
    totals = mask_3ch_hwd.sum(axis=(0,1,2))  # per-slice tumor voxels along D
    return int(np.argmax(totals))


# Minimal 3D U-Net + loader

In [5]:
class DoubleConv3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=4, out_channels=3, base_ch=64):
        super().__init__()
        # Encoder
        self.enc1 = DoubleConv3D(in_channels, base_ch)         # 4 -> 64
        self.pool1 = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))
        self.enc2 = DoubleConv3D(base_ch, base_ch*2)           # 64 -> 128
        self.pool2 = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))
        self.enc3 = DoubleConv3D(base_ch*2, base_ch*4)         # 128 -> 256
        self.pool3 = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))

        # Bottleneck
        self.bottleneck = DoubleConv3D(base_ch*4, base_ch*8)   # 256 -> 512

        # Decoder
        self.up3 = nn.ConvTranspose3d(base_ch*8, base_ch*4, kernel_size=(1,2,2), stride=(1,2,2))
        self.dec3 = DoubleConv3D(base_ch*8, base_ch*4)         # concat 256 + 256 -> 256
        self.up2 = nn.ConvTranspose3d(base_ch*4, base_ch*2, kernel_size=(1,2,2), stride=(1,2,2))
        self.dec2 = DoubleConv3D(base_ch*4, base_ch*2)         # concat 128 + 128 -> 128
        self.up1 = nn.ConvTranspose3d(base_ch*2, base_ch,     kernel_size=(1,2,2), stride=(1,2,2))
        self.dec1 = DoubleConv3D(base_ch*2, base_ch)           # concat 64 + 64 -> 64

        # Output
        self.out_conv = nn.Conv3d(base_ch, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        e3 = self.enc3(p2)
        p3 = self.pool3(e3)

        # Bottleneck
        b = self.bottleneck(p3)

        # Decoder
        u3 = self.up3(b)
        if u3.shape[2:] != e3.shape[2:]:
            u3 = self._center_crop_or_pad(u3, e3.shape[2:])
        d3 = self.dec3(torch.cat([u3, e3], dim=1))

        u2 = self.up2(d3)
        if u2.shape[2:] != e2.shape[2:]:
            u2 = self._center_crop_or_pad(u2, e2.shape[2:])
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[2:] != e1.shape[2:]:
            u1 = self._center_crop_or_pad(u1, e1.shape[2:])
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        out = self.out_conv(d1)  # (B,3,D,H,W) raw logits
        return out

    @staticmethod
    def _center_crop_or_pad(x, target_spatial):
        _, _, D, H, W = x.shape
        tD, tH, tW = target_spatial
        dD = tD - D
        dH = tH - H
        dW = tW - W

        padD = (max(0, dD//2), max(0, dD - dD//2))
        padH = (max(0, dH//2), max(0, dH - dH//2))
        padW = (max(0, dW//2), max(0, dW - dW//2))
        if any(p > 0 for p in (*padD, *padH, *padW)):
            x = F.pad(x, (padW[0], padW[1], padH[0], padH[1], padD[0], padD[1]))

        _, _, D, H, W = x.shape
        sD = max(0, (D - tD)//2)
        sH = max(0, (H - tH)//2)
        sW = max(0, (W - tW)//2)
        x = x[:, :, sD:sD+tD, sH:sH+tH, sW:sW+tW]
        return x

def load_model():
    model = UNet3D(in_channels=4, out_channels=3, base_ch=64).to(device)
    model = model.to(memory_format=torch.channels_last_3d)
    ckpt = os.path.join(CHECKPOINT_DIR, "best_by_val_dice.pth")
    if os.path.exists(ckpt):
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state["model_state"] if "model_state" in state else state)
        print("Loaded checkpoint:", ckpt)
    else:
        print("NOTE: best_by_val_dice.pth not found; inference will use random weights.")
    model.eval()
    return model

_model = load_model()


Loaded checkpoint: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/best_by_val_dice.pth


# Modalities overview

In [12]:
import os, json
import numpy as np
import matplotlib.pyplot as plt

# ---- adjustable knobs ----
TITLE_FONTSIZE = 16        # bigger titles
TITLE_PAD      = 10
AUTO_THRESH    = 0.05      # % of max used to find brain bbox on FLAIR
AUTO_MARGIN    = 12        # extra pixels kept around the auto bbox
TRIM_LEFT      = 6         # manual pixels to chop from the left (after auto-crop)
TRIM_RIGHT     = 6         # manual pixels to chop from the right (after auto-crop)

def _pick(d, opts):
    for k in opts:
        if k in d: return k
    raise KeyError(f"None of {opts} in {list(d.keys())}")

def _auto_bbox(img, thresh=AUTO_THRESH, margin=AUTO_MARGIN):
    """Return (y0,y1,x0,x1) bounding box around high-intensity area on FLAIR."""
    m = img > (img.max() * thresh)
    if not np.any(m):
        return (0, img.shape[0], 0, img.shape[1])
    ys, xs = np.where(m)
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    y0 = max(y0 - margin, 0)
    y1 = min(y1 + margin, img.shape[0])
    x0 = max(x0 - margin, 0)
    x1 = min(x1 + margin, img.shape[1])
    return (y0, y1, x0, x1)

def _apply_bbox_and_trim(arr, y0, y1, x0, x1, trim_left=0, trim_right=0):
    """Crop to bbox, then trim fixed columns on left/right. Works on 2D or 3D last-dim=4 RGBA."""
    sub = arr[y0:y1, x0:x1] if arr.ndim == 2 else arr[y0:y1, x0:x1, :]
    if trim_left or trim_right:
        x1_new = sub.shape[1] - trim_right
        sub = sub[:, trim_left:x1_new] if arr.ndim == 2 else sub[:, trim_left:x1_new, :]
    return sub

def fig_modalities_overview(subject_id, out_base="fig01_modalities_overview"):
    """
    1x5 row: T1 | T1Gd | T2 | FLAIR | Ground Truth
    GT drawn as one opaque RGBA overlay:
      WT = green, TC = yellow, ET = red (ET > TC > WT priority).
    Shared vmin/vmax across modalities. Auto-crops using FLAIR, then trims left/right.
    """
    vols, seg = load_modalities(subject_id)  # seg: (H,W,D) labels {0,1,2,3}

    # slice: prefer most ET, else most tumor
    et_area = (seg == 3).sum(axis=(0,1))
    z = int(np.argmax(et_area)) if et_area.max() > 0 else int(np.argmax((seg > 0).sum(axis=(0,1))))
    SL = seg[:, :, z]

    # modality keys
    k_T1    = _pick(vols, ["t1n","t1","t1_native"])
    k_T1Gd  = _pick(vols, ["t1c","t1gd","t1ce"])
    k_T2    = _pick(vols, ["t2w","t2"])
    k_FLAIR = _pick(vols, ["t2f","flair"])

    # slices
    s_T1, s_T1Gd, s_T2, s_FLAIR = (
        vols[k_T1][:,:,z], vols[k_T1Gd][:,:,z], vols[k_T2][:,:,z], vols[k_FLAIR][:,:,z]
    )

    # shared grayscale window
    stack = np.stack([s_T1, s_T1Gd, s_T2, s_FLAIR]).astype(np.float32)
    lo, hi = np.percentile(stack, 1), np.percentile(stack, 99)

    # masks (0/1/2/3)
    ET = (SL == 3)
    TC = np.logical_or(SL == 1, ET)
    WT = np.logical_or(SL == 2, TC)

    # RGBA overlay (opaque)
    H, W = SL.shape
    rgba = np.zeros((H, W, 4), dtype=np.float32)
    rgba[WT] = (0.05, 0.70, 0.20, 1.0)   # green
    rgba[TC] = (0.98, 0.85, 0.10, 1.0)   # yellow
    rgba[ET] = (0.85, 0.10, 0.10, 1.0)   # red

    # --- auto-crop bbox from FLAIR, then apply manual trim to all panels consistently
    y0, y1, x0, x1 = _auto_bbox(s_FLAIR, thresh=AUTO_THRESH, margin=AUTO_MARGIN)
    s_T1_c     = _apply_bbox_and_trim(s_T1,     y0, y1, x0, x1, TRIM_LEFT, TRIM_RIGHT)
    s_T1Gd_c   = _apply_bbox_and_trim(s_T1Gd,   y0, y1, x0, x1, TRIM_LEFT, TRIM_RIGHT)
    s_T2_c     = _apply_bbox_and_trim(s_T2,     y0, y1, x0, x1, TRIM_LEFT, TRIM_RIGHT)
    s_FLAIR_c  = _apply_bbox_and_trim(s_FLAIR,  y0, y1, x0, x1, TRIM_LEFT, TRIM_RIGHT)
    rgba_c     = _apply_bbox_and_trim(rgba,     y0, y1, x0, x1, TRIM_LEFT, TRIM_RIGHT)

    titles = ["T1","T1Gd","T2","FLAIR","Ground Truth"]
    imgs   = [s_T1_c, s_T1Gd_c, s_T2_c, s_FLAIR_c]

    # plot
    fig, axes = plt.subplots(1, 5, figsize=(18, 5))  # a bit taller for bigger titles
    fig.patch.set_facecolor("white")

    for ax, img, title in zip(axes[:4], imgs, titles[:4]):
        ax.imshow(img, cmap="gray", vmin=lo, vmax=hi, interpolation="nearest")
        ax.set_title(title, fontsize=TITLE_FONTSIZE, pad=TITLE_PAD, fontweight="bold")
        ax.axis("off")

    ax = axes[4]
    ax.imshow(s_FLAIR_c, cmap="gray", vmin=lo, vmax=hi, interpolation="nearest")
    ax.imshow(rgba_c, interpolation="nearest")
    ax.set_title(titles[4], fontsize=TITLE_FONTSIZE, pad=TITLE_PAD, fontweight="bold")
    ax.axis("off")

    plt.subplots_adjust(wspace=0.02)
    os.makedirs(FIG_DIR, exist_ok=True)
    pdf = os.path.join(FIG_DIR, out_base + ".pdf")
    png = os.path.join(FIG_DIR, out_base + ".png")
    plt.savefig(pdf, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.savefig(png, bbox_inches="tight", facecolor=fig.get_facecolor(), dpi=300)
    plt.close()
    print("Saved:", pdf)
    print("Saved:", png)

# ---- Run it now ----
splits_path = os.path.join(CHECKPOINT_DIR, "splits.json")
sp = json.load(open(splits_path))
assert len(sp.get("test", [])) > 0, "No test subjects found in splits.json"

subject_id = "BraTS-GLI-00000-000"  # or sp["test"][0]
print("Modalities overview subject:", subject_id)
fig_modalities_overview(subject_id)


Modalities overview subject: BraTS-GLI-00000-000
Saved: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/paper_figures/fig01_modalities_overview.pdf
Saved: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/paper_figures/fig01_modalities_overview.png


# Training dynamics (validation-only: Loss, Dice, IoU, HD95)

In [14]:
from matplotlib.ticker import MultipleLocator

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"]  = 42

# Read metrics
df = pd.read_csv(METRICS_CSV)
assert {"epoch","phase","loss","dice_mean","iou_mean","hd95_mean"}.issubset(df.columns)
val = df[df["phase"] == "val"].copy()

# 2x2 layout
fig, axs = plt.subplots(2, 2, figsize=(10, 8))
axes = axs.ravel()

plots = [
    ("loss",      "Loss"),
    ("dice_mean", "Dice"),
    ("iou_mean",  "IoU"),
    ("hd95_mean", "HD95"),
]

for ax, (col, title) in zip(axes, plots):
    ax.plot(val["epoch"], val[col], linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    #ax.set_ylabel("Value")
    ax.xaxis.set_major_locator(MultipleLocator(5))  # tick every 5 epochs
    ax.grid(True, linestyle=":", linewidth=0.7, alpha=0.6)

plt.tight_layout()
os.makedirs(FIG_DIR, exist_ok=True)

out_png = os.path.join(FIG_DIR, "fig02_training_dynamics_val_only.png")
out_pdf = os.path.join(FIG_DIR, "fig02_training_dynamics_val_only.pdf")
plt.savefig(out_png, bbox_inches="tight", dpi=300)
plt.savefig(out_pdf, bbox_inches="tight")
plt.close()

print("Saved:", out_png)
print("Saved:", out_pdf)


Saved: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/paper_figures/fig02_training_dynamics_val_only.png
Saved: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/paper_figures/fig02_training_dynamics_val_only.pdf


# Pick best/median/worst subjects by mean Dice

In [15]:
per_case = pd.read_csv(TEST_METRICS_PER_SUBJECT)
assert "subject" in per_case.columns
for c in ["dice_wt","dice_tc","dice_et"]:
    assert c in per_case.columns, f"Missing column {c} in TEST_METRICS_PER_SUBJECT."

per_case["dice_mean"] = per_case[["dice_wt","dice_tc","dice_et"]].mean(axis=1)
per_case_sorted = per_case.sort_values("dice_mean", ascending=False).reset_index(drop=True)

best_subj   = per_case_sorted.iloc[0]["subject"]
median_subj = per_case_sorted.iloc[len(per_case_sorted)//2]["subject"]
worst_subj  = per_case_sorted.iloc[-1]["subject"]

print("Best   :", best_subj)
print("Median :", median_subj)
print("Worst  :", worst_subj)


Best   : BraTS-GLI-01401-000
Median : BraTS-GLI-00590-000
Worst  : BraTS-GLI-00540-000


# Prediction helper

In [16]:
def predict_subject(subject_id, model=_model):
    """
    Returns (pred_3ch_hwd, gt_3ch_hwd, t2f_hwd) with pred/gt in {0,1}.
    If PRED_DIR has <subject>_pred.nii.gz, it will be used; else run quick forward pass.
    """
    vols, seg = load_modalities(subject_id)
    gt_3ch = brats_channels_from_seg(seg)   # (3,H,W,D)
    t2f = vols["t2f"]                       # (H,W,D)

    # If you saved predictions as per-voxel labels 0..k
    pred_path = os.path.join(PRED_DIR, f"{subject_id}_pred.nii.gz")
    if os.path.exists(pred_path):
        p = nib.load(pred_path).get_fdata()
        if p.ndim == 3:
            pred_3ch = brats_channels_from_seg(p.astype(np.int16))
        else:
            pred_3ch = p  # assume already 3-channel HWD
    else:
        assert model is not None, "Model not loaded and predictions not available."
        # Build (1,4,D,H,W) tensor in DHW order
        t1c = torch.from_numpy(np.transpose(vols["t1c"], (2,0,1))).unsqueeze(0)
        t1n = torch.from_numpy(np.transpose(vols["t1n"], (2,0,1))).unsqueeze(0)
        t2w = torch.from_numpy(np.transpose(vols["t2w"], (2,0,1))).unsqueeze(0)
        t2f_t = torch.from_numpy(np.transpose(vols["t2f"], (2,0,1))).unsqueeze(0)
        img = torch.cat([t1c,t1n,t2w,t2f_t], dim=0).unsqueeze(0).to(device).to(memory_format=torch.channels_last_3d)

        with torch.inference_mode(), torch.amp.autocast(device_type=device if device=="cuda" else "cpu", enabled=(device=="cuda")):
            logits = model(img)
            probs = torch.sigmoid(logits)[0].cpu().numpy()  # (3,D,H,W)
        pred_bin = (probs > 0.5).astype(np.uint8)
        pred_3ch = np.transpose(pred_bin, (0,2,3,1))        # (3,H,W,D)

    return pred_3ch, gt_3ch, t2f


# Evaluation grid (best/median/worst)

In [20]:
# --- same colors as modalities (opaque) ---
COL_WT = (0.05, 0.70, 0.20, 1.0)  # green
COL_TC = (0.98, 0.85, 0.10, 1.0)  # yellow
COL_ET = (0.85, 0.10, 0.10, 1.0)  # red

def _rgba_from_wt_tc_et(wt, tc, et, disjoint=False):
    """
    Build an RGBA overlay from WT/TC/ET masks.
    - If disjoint=True: draws WT\TC (green), TC\ET (yellow), ET (red) (clear nesting).
    - If disjoint=False: paints priority WT< TC< ET (ET over TC over WT).
    """
    wt = (wt > 0.5) if wt.dtype != bool else wt
    tc = (tc > 0.5) if tc.dtype != bool else tc
    et = (et > 0.5) if et.dtype != bool else et
    H, W = wt.shape
    rgba = np.zeros((H, W, 4), dtype=np.float32)

    if disjoint:
        green  = np.logical_and(wt, ~tc)  # WT \ TC
        yellow = np.logical_and(tc, ~et)  # TC \ ET
        red    = et
        rgba[green]  = COL_WT
        rgba[yellow] = COL_TC
        rgba[red]    = COL_ET
    else:
        # priority: WT then TC then ET (ET on top)
        rgba[wt] = COL_WT
        rgba[tc] = COL_TC
        rgba[et] = COL_ET
    return rgba

def evaluation_grid_reference_style(subjects, out_png, disjoint=False):
    fig, axes = plt.subplots(3, 3, figsize=(12, 10))
    fig.patch.set_facecolor("white")
    row_titles = ["Best (by Dice mean)", "Median", "Worst (by Dice mean)"]

    for r, sid in enumerate(subjects):
        pred, gt, t2f = predict_subject(sid)  # pred/gt shape: (3,H,W,D) in order WT,TC,ET
        z = slice_with_max_tumor(gt)

        # shared window per subject (1–99% on this subject's volume)
        lo, hi = np.percentile(t2f, [1, 99])

        # slice masks (bool or 0/1); also robust to soft predictions
        wt, tc, et         = gt[0, :, :, z], gt[1, :, :, z], gt[2, :, :, z]
        p_wt, p_tc, p_et   = pred[0, :, :, z], pred[1, :, :, z], pred[2, :, :, z]

        # build RGBA overlays with SAME COLORS as modalities
        rgba_gt = _rgba_from_wt_tc_et(wt,  tc,  et,  disjoint=disjoint)
        rgba_pr = _rgba_from_wt_tc_et(p_wt, p_tc, p_et, disjoint=disjoint)

        # Left: FLAIR
        ax = axes[r, 0]
        show_bg(ax, t2f[:, :, z], lo, hi)
        ax.set_title(f"{row_titles[r]}\n{sid} — T2-FLAIR")

        # Middle: Ground truth
        ax = axes[r, 1]
        show_bg(ax, t2f[:, :, z], lo, hi)
        ax.set_title("Ground truth")
        ax.imshow(rgba_gt, interpolation="nearest")

        # Right: Prediction
        ax = axes[r, 2]
        show_bg(ax, t2f[:, :, z], lo, hi)
        ax.set_title("Prediction")
        ax.imshow(rgba_pr, interpolation="nearest")

    plt.tight_layout()
    outp = os.path.join(FIG_DIR, out_png)
    plt.savefig(outp, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(); print("Saved:", outp)

# usage (set disjoint=True if you want the green/yellow/red shells to be visually non-overlapping):
subjects = [best_subj, median_subj, worst_subj]
evaluation_grid_reference_style(subjects, "fig03_evaluation_best_median_worst.pdf", disjoint=False)


<>:9: SyntaxWarning: invalid escape sequence '\T'
<>:9: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipython-input-173060950.py:9: SyntaxWarning: invalid escape sequence '\T'
  - If disjoint=True: draws WT\TC (green), TC\ET (yellow), ET (red) (clear nesting).


Saved: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/paper_figures/fig03_evaluation_best_median_worst.pdf


# Summary boxplots (Dice, IoU, HD95)

In [ ]:
# -------- Summary boxplots (WT, TC, ET) --------
pc = pd.read_csv(TEST_METRICS_PER_SUBJECT)

def nice_box(ax, data, labels, title, ylim=None, ylab=None):
    ax.boxplot(
        data,
        labels=labels,
        notch=True,
        showmeans=True,
        meanprops=dict(marker="^", markersize=6),
        whiskerprops=dict(linewidth=1.25),
        boxprops=dict(linewidth=1.25),
        capprops=dict(linewidth=1.25),
        medianprops=dict(linewidth=1.25),
    )
    ax.set_title(title)
    if ylim: ax.set_ylim(*ylim)
    if ylab: ax.set_ylabel(ylab)

fig, axes = plt.subplots(3, 1, figsize=(7, 14))  # tall layout

# --- Dice ---
nice_box(
    axes[0],
    [pc["dice_wt"], pc["dice_tc"], pc["dice_et"]],   # WT, TC, ET
    ["WT", "TC", "ET"],
    "Dice per subject",
    ylim=(0, 1),
    ylab="Dice"
)

# --- IoU ---
nice_box(
    axes[1],
    [pc["iou_wt"], pc["iou_tc"], pc["iou_et"]],      # WT, TC, ET
    ["WT", "TC", "ET"],
    "IoU per subject",
    ylim=(0, 1),
    ylab="IoU"
)

# --- HD95 ---
ymax = max(
    10,
    np.nanpercentile(
        pd.concat([pc["hd95_wt"], pc["hd95_tc"], pc["hd95_et"]]), 97
    )
)
nice_box(
    axes[2],
    [pc["hd95_wt"], pc["hd95_tc"], pc["hd95_et"]],   # WT, TC, ET
    ["WT", "TC", "ET"],
    "HD95 per subject (voxels)",
    ylim=(0, ymax),
    ylab="HD95 (voxels)"
)

plt.tight_layout()
outp = os.path.join(FIG_DIR, "fig04_summary_boxplots.pdf")
plt.savefig(outp, bbox_inches="tight"); plt.close()
print("Saved:", outp)


/tmp/ipython-input-3411506935.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(
/tmp/ipython-input-3411506935.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(
/tmp/ipython-input-3411506935.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(


Saved: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/paper_figures/fig04_summary_boxplots.pdf


# Inference time

In [6]:
# Config
TIMING_SAMPLE = None          # set None to run on ALL test subjects
WARMUP_FOR_DEVICE = 3       # one-time warmup (not per subject)

# Sample subjects
sp = json.load(open(os.path.join(CHECKPOINT_DIR,"splits.json")))
test_subjects = sp["test"]
if TIMING_SAMPLE is not None:
    rng = np.random.default_rng(1337)
    test_subjects = list(rng.choice(test_subjects, size=min(TIMING_SAMPLE, len(test_subjects)), replace=False))

model = _model if _model is not None else load_model()
model.eval()

# One-time warmup (no disk I/O)
vols0, _ = load_modalities(test_subjects[0])
t1c = torch.from_numpy(np.transpose(vols0["t1c"], (2,0,1))).unsqueeze(0)
t1n = torch.from_numpy(np.transpose(vols0["t1n"], (2,0,1))).unsqueeze(0)
t2w = torch.from_numpy(np.transpose(vols0["t2w"], (2,0,1))).unsqueeze(0)
t2f = torch.from_numpy(np.transpose(vols0["t2f"], (2,0,1))).unsqueeze(0)
img0 = torch.cat([t1c,t1n,t2w,t2f], dim=0).unsqueeze(0).to(device).to(memory_format=torch.channels_last_3d)
with torch.inference_mode(), torch.amp.autocast(device_type=device if device=="cuda" else "cpu", enabled=(device=="cuda")):
    for _ in range(WARMUP_FOR_DEVICE):
        _ = model(img0)
        if device=="cuda": torch.cuda.synchronize()

rows = []
for sid in tqdm(test_subjects, desc="Inference timing"):
    t0 = time.perf_counter()
    vols, _ = load_modalities(sid)                     # read + z-score
    t1 = time.perf_counter()

    t1c = torch.from_numpy(np.transpose(vols["t1c"], (2,0,1))).unsqueeze(0)
    t1n = torch.from_numpy(np.transpose(vols["t1n"], (2,0,1))).unsqueeze(0)
    t2w = torch.from_numpy(np.transpose(vols["t2w"], (2,0,1))).unsqueeze(0)
    t2f = torch.from_numpy(np.transpose(vols["t2f"], (2,0,1))).unsqueeze(0)
    img = torch.cat([t1c,t1n,t2w,t2f], dim=0).unsqueeze(0).to(device).to(memory_format=torch.channels_last_3d)
    t2p = time.perf_counter()

    with torch.inference_mode(), torch.amp.autocast(device_type=device if device=="cuda" else "cpu", enabled=(device=="cuda")):
        t2 = time.perf_counter()
        _ = model(img)
        if device=="cuda": torch.cuda.synchronize()
        t3 = time.perf_counter()

    rows.append({
        "subject": sid,
        "time_io_norm": t1 - t0,
        "time_build_h2d": t2p - t1,
        "time_forward": t3 - t2,
        "time_end_to_end": t3 - t0
    })

proxy = pd.DataFrame(rows)
proxy_csv = os.path.join(CHECKPOINT_DIR, "proxy_time_breakdown_sample.csv")

proxy.to_csv(proxy_csv, index=False); print("Saved:", proxy_csv)
display(proxy.describe())

def scatter_box(series, title, fname):
    fig, ax = plt.subplots(figsize=(6,4))
    ax.boxplot(series.values, vert=True, showmeans=True)
    ax.scatter(np.random.normal(1, 0.02, size=len(series)), series.values, s=8, alpha=0.45)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylabel("Seconds per 3D volume", fontsize=11)
    ax.set_xticks([])
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, fname)
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print("Saved:", path)

n_subjects = len(proxy)
scatter_box(
    proxy["time_end_to_end"],
    f"End-to-End Inference Time (n={n_subjects}, device=gpu)",
    "fig05_end_to_end_inference_time.pdf"
)

scatter_box(
    proxy["time_forward"],
    f"Model-Only Forward Pass (n={n_subjects}, device=gpu)",
    "fig05_model_only_forward_time.pdf"
)

Inference timing: 100%|██████████| 188/188 [38:30<00:00, 12.29s/it]


Saved: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/proxy_time_breakdown_sample.csv


,time_io_norm,time_build_h2d,time_forward,time_end_to_end
count,188.000000,188.000000,188.000000,188.000000
mean,11.616193,0.090208,0.582039,12.288558
std,2.037623,0.009872,0.006549,2.038823
min,1.489971,0.076876,0.563143,2.149238
25%,10.485600,0.078806,0.576303,11.155883
50%,11.202198,0.091936,0.582013,11.863368
75%,12.424150,0.098772,0.587612,13.097611
max,22.752205,0.107892,0.594983,23.439418


Saved: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/paper_figures/fig05_end_to_end_inference_time.pdf
Saved: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/paper_figures/fig05_model_only_forward_time.pdf


# Comparison table

In [ ]:
summary = pd.read_csv(TEST_METRICS_CSV).iloc[0]

your_row = {
    "Model": "This work (3D U-Net)",
    "Dice_ET":  round(float(summary["dice_et"]), 4),
    "Dice_TC":  round(float(summary["dice_tc"]), 4),
    "Dice_WT":  round(float(summary["dice_wt"]), 4),
    "Dice_Mean": round(float(summary["dice_mean"]), 4),
    "IoU_Mean":  round(float(summary["iou_mean"]), 4),
    "HD95_Mean": round(float(summary["hd95_mean"]), 2),
}

# Fill exact numbers from the papers you’re benchmarking against:
literature_rows = [
    {"Model": "SegResNet-DS (2023)", "Dice_ET": np.nan, "Dice_TC": np.nan, "Dice_WT": np.nan, "Dice_Mean": np.nan, "IoU_Mean": np.nan, "HD95_Mean": np.nan},
    {"Model": "MedNeXt (2023)",      "Dice_ET": np.nan, "Dice_TC": np.nan, "Dice_WT": np.nan, "Dice_Mean": np.nan, "IoU_Mean": np.nan, "HD95_Mean": np.nan},
    {"Model": "Top Ensemble (2023)", "Dice_ET": np.nan, "Dice_TC": np.nan, "Dice_WT": np.nan, "Dice_Mean": np.nan, "IoU_Mean": np.nan, "HD95_Mean": np.nan},
]

comp_df = pd.DataFrame([your_row] + literature_rows)
display(comp_df)

# Save LaTeX table for your paper
tex_path = os.path.join(FIG_DIR, "table_results_comparison.tex")
with open(tex_path, "w") as f:
    f.write(comp_df.to_latex(index=False, float_format="%.4f", na_rep="--"))
print("Saved LaTeX table:", tex_path)


,Model,Dice_ET,Dice_TC,Dice_WT,Dice_Mean,IoU_Mean,HD95_Mean
0,This work (3D U-Net),0.8703,0.8931,0.9246,0.896,0.8379,3.85
1,SegResNet-DS (2023),NaN,NaN,NaN,NaN,NaN,NaN
2,MedNeXt (2023),NaN,NaN,NaN,NaN,NaN,NaN
3,Top Ensemble (2023),NaN,NaN,NaN,NaN,NaN,NaN


Saved LaTeX table: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/paper_figures/table_results_comparison.tex
